# Clase 211 — Polars: lazy API, streaming, benchmarks

Requiere: `pip install polars pyarrow pandas duckdb`.

In [ ]:
import polars as pl, pandas as pd, time, tempfile, os
from pathlib import Path

WORK = Path(tempfile.gettempdir()) / 'polars_demo'
WORK.mkdir(exist_ok=True)
print('Polars version:', pl.__version__)

## 1. Dataset sintético — 5M filas

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
N = 5_000_000
df_pl = pl.DataFrame({
    'zone_id': rng.integers(0, 100, N),
    'fare':    rng.uniform(5, 100, N),
    'tip':     rng.uniform(0, 20, N),
    'date':    pl.date(2024, 1, 1) + pl.duration(days=pl.Series(rng.integers(0, 60, N))),
    'borough': rng.choice(['Manhattan', 'Brooklyn', 'Queens', 'Bronx'], N),
})

pq = WORK / 'trips.parquet'
df_pl.write_parquet(pq)
print(f'parquet: {pq.stat().st_size / 1024 / 1024:.1f} MB, {N:,} filas')

## 2. Pandas vs Polars eager vs Polars lazy

In [ ]:
# Misma query: agregado por borough con filtro
def bench(name, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    print(f'{name:25} {dt*1000:>8.1f} ms')
    return out

bench('pandas', lambda: (pd.read_parquet(pq).query('fare > 30').groupby('borough')
                          .agg(avg_fare=('fare', 'mean'), n=('fare', 'size'))))

bench('polars eager', lambda: (pl.read_parquet(pq).filter(pl.col('fare') > 30)
                                .group_by('borough').agg(pl.col('fare').mean().alias('avg_fare'),
                                                          pl.len().alias('n'))))

bench('polars lazy', lambda: (pl.scan_parquet(pq).filter(pl.col('fare') > 30)
                               .group_by('borough').agg(pl.col('fare').mean().alias('avg_fare'),
                                                         pl.len().alias('n'))
                               .collect()))

## 3. Optimizaciones del query planner

In [ ]:
q = (pl.scan_parquet(pq)
     .filter(pl.col('borough') == 'Manhattan')
     .filter(pl.col('fare').is_between(10, 50))
     .select('zone_id', 'fare', 'tip')
     .group_by('zone_id').agg(pl.col('fare').mean(), pl.col('tip').mean()))

print('=== Plan optimizado ===')
print(q.explain())
print('\n→ Observá: PROJECT solo 4 columnas (column pruning),')
print('  filter pushed down al PARQUET SCAN (predicate pushdown).')

## 4. Streaming engine (datasets > RAM)

In [ ]:
# Streaming es importante con datasets que NO caben en RAM.
# Acá funciona igual; en datasets de 100+ GB es la diferencia entre OK y OOM.
t0 = time.perf_counter()
out = q.collect(engine='streaming')
print(f'streaming engine: {(time.perf_counter() - t0) * 1000:.1f} ms')
print(out.head())

## 5. Window functions con `over()`

In [ ]:
# Rolling mean de fare por borough, ordenado por date
result = (df_pl.sort('date')
          .with_columns([
              pl.col('fare').rolling_mean(window_size=10000).over('borough').alias('rolling_avg_fare'),
              (pl.col('fare') - pl.col('fare').mean().over('borough')).alias('fare_dev_from_borough_mean'),
          ]))
result.select('date', 'borough', 'fare', 'rolling_avg_fare', 'fare_dev_from_borough_mean').head()

## 6. Interop con DuckDB (zero-copy via Arrow)

In [ ]:
import duckdb
con = duckdb.connect()

# DuckDB lee Polars directo (Arrow zero-copy)
sql_result = con.execute('''
    SELECT borough, AVG(fare) AS avg_fare, COUNT(*) AS n
    FROM df_pl
    WHERE fare > 30
    GROUP BY borough
    ORDER BY avg_fare DESC
''').pl()   # devolver como Polars DataFrame
print(sql_result)
con.close()

## Ejercicio guiado

1. Migrá un script pandas tuyo a Polars. Medí speedup. Casos donde NO sea más rápido: documentar por qué.
2. Usá `.explain()` antes y después de cambiar el orden de filters/selects — observá que el optimizer da el mismo plan.
3. Generá un parquet de 5 GB (loop generando chunks) y procesalo con `engine='streaming'`. Medí RAM peak con `psutil`.
4. Combiná Polars + DuckDB: feature engineering en Polars, query analítica final en SQL DuckDB sobre el resultado.
5. Bonus: integrá Polars en un flow Prefect (Clase 209).

## Conclusiones

- Polars eager ≈ 3-10× pandas; Polars lazy ≈ 5-30× pandas en pipelines reales.
- Lazy es para producción; eager para REPL.
- Streaming engine permite out-of-core sin saltar a Spark.
- Arrow zero-copy hace que el interop con DuckDB / pandas / PyArrow sea gratis.

## ✅ Soluciones de los ejercicios

Polars **sí** está instalado, así que estas soluciones se ejecutan de verdad. Trabajamos
sobre un parquet sintético (sin internet) para medir *eager vs lazy*, traducir un pipeline
pandas, ver el *predicate pushdown* en el plan, y pasar el resultado a DuckDB vía Arrow.

### Ejercicio 1 — Eager vs Lazy benchmark

`pl.read_parquet` (eager: carga todo y agrega) vs `pl.scan_parquet(...).collect()` (lazy:
arma un plan y lo optimiza). Con datos chicos la diferencia es mínima; el punto es que el
resultado es idéntico y el lazy escala a datasets >RAM.

In [ ]:
import tempfile, time
from pathlib import Path
import numpy as np, polars as pl

WORK = Path(tempfile.gettempdir()) / "polars_sim"; WORK.mkdir(exist_ok=True)
rng = np.random.default_rng(0)
N = 500_000
df0 = pl.DataFrame({
    "category": rng.integers(0, 20, N),
    "date": rng.integers(0, 30, N),
    "fare": rng.gamma(3.0, 4.0, N),
})
path = WORK / "trips.parquet"
df0.write_parquet(path)

def eager():
    return pl.read_parquet(path).group_by("category").agg(pl.col("fare").mean()).sort("category")

def lazy():
    return (pl.scan_parquet(path).group_by("category").agg(pl.col("fare").mean())
            .sort("category").collect())

t0 = time.perf_counter(); r_eager = eager(); t_eager = time.perf_counter() - t0
t0 = time.perf_counter(); r_lazy = lazy();  t_lazy = time.perf_counter() - t0
print(f"eager: {t_eager*1000:.1f} ms | lazy: {t_lazy*1000:.1f} ms")

assert r_eager.equals(r_lazy), "mismo resultado por ambos caminos"
assert r_eager.height == 20
print("OK ejercicio 1 — eager y lazy dan el mismo resultado")

### Ejercicio 2 — Pandas → Polars

Traducimos un pipeline pandas típico (`groupby().agg()`, `apply`) a expresiones Polars.
`.apply` fila-a-fila se reemplaza por **expresiones vectorizadas**, mucho más rápidas.

In [ ]:
import pandas as pd

pdf = df0.to_pandas()

# --- versión pandas ---------------------------------------------------------
res_pd = (pdf.assign(fare_tax=pdf["fare"] * 1.1)          # .apply evitado: vectorizado
             .groupby("category")["fare_tax"].mean()
             .reset_index()
             .sort_values("category"))

# --- versión Polars (expresiones) ------------------------------------------
res_pl = (df0.with_columns((pl.col("fare") * 1.1).alias("fare_tax"))
             .group_by("category").agg(pl.col("fare_tax").mean())
             .sort("category"))

merged = res_pl.to_pandas().rename(columns={"fare_tax": "pl"}).assign(pd=res_pd["fare_tax"].values)
assert np.allclose(merged["pl"], merged["pd"]), "misma agregación en pandas y Polars"
print(merged.head())
print("OK ejercicio 2 — pipeline pandas traducido a expresiones Polars, mismo resultado")

### Ejercicio 3 — Streaming

El motor *streaming* procesa el parquet **por chunks** en vez de cargarlo entero, bajando el
pico de RAM. En datasets >RAM esa es la diferencia entre correr y morir con OOM. Acá el
dataset cabe en memoria, así que solo mostramos que ambos caminos dan lo mismo.

In [ ]:
plan = pl.scan_parquet(path).group_by("category").agg(pl.col("fare").sum()).sort("category")

normal = plan.collect()
try:
    streamed = plan.collect(engine="streaming")   # motor por chunks (API nueva)
    modo = "engine=streaming"
except TypeError:
    streamed = plan.collect(streaming=True)        # fallback API previa
    modo = "streaming=True"

print("modo streaming usado:", modo)
a = normal.sort("category"); b = streamed.sort("category")
assert a["category"].to_list() == b["category"].to_list()
assert np.allclose(a["fare"].to_numpy(), b["fare"].to_numpy())  # mismo resultado, menor pico de RAM
print("OK ejercicio 3 — agregación en streaming = mismo resultado")

### Ejercicio 4 — Predicate pushdown explícito

Con `scan_parquet(...).filter(...).select(...)`, Polars **empuja el filtro y la proyección
al lectura del parquet**: lee menos filas y menos columnas. Lo confirmamos leyendo el plan
optimizado con `.explain()`.

In [ ]:
lazy_q = (pl.scan_parquet(path)
            .filter(pl.col("date") == 15)
            .select(["fare"]))

plan_txt = lazy_q.explain()          # plan físico optimizado
print(plan_txt)

low = plan_txt.lower()
# el plan debe mostrar que el filtro/selección bajaron al scan del parquet
assert "parquet" in low
assert ("selection" in low) or ("filter" in low) or ("predicate" in low)
result = lazy_q.collect()
assert result.columns == ["fare"], "solo se proyectó la columna pedida"
print("filas tras pushdown:", result.height)
print("OK ejercicio 4 — filtro y proyección empujados al scan (predicate/projection pushdown)")

### Ejercicio 5 — Polars + DuckDB vía Arrow

Hacemos la parte de manipulación en Polars y pasamos el resultado a DuckDB **sin copiar**
(vía Arrow) para una consulta SQL. `to_arrow()` + replacement scan de DuckDB.

In [ ]:
import duckdb

# Polars prepara los datos...
prep = df0.with_columns((pl.col("fare") * 1.1).alias("fare_tax"))
arrow_tbl = prep.to_arrow()          # zero-copy handoff a DuckDB

# ...DuckDB corre el SQL complejo sobre la tabla Arrow (la referencia por nombre)
sql = (
    "SELECT category, COUNT(*) AS n, ROUND(AVG(fare_tax), 3) AS avg_tax "
    "FROM arrow_tbl GROUP BY category "
    "HAVING COUNT(*) > 1000 ORDER BY avg_tax DESC LIMIT 5"
)
top = duckdb.sql(sql).pl()           # de vuelta a Polars
print(top)

assert top.height <= 5 and "avg_tax" in top.columns
assert top["n"].min() > 1000, "el HAVING filtró categorías chicas"
print("OK ejercicio 5 — Polars -> Arrow -> DuckDB SQL -> Polars")